In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, cohen_kappa_score, balanced_accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATA_DIR = "/kaggle/working/Final_Data_CLAHE"

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR,"train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR,"val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR,"test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)
test_loader  = DataLoader(test_dataset, batch_size=8)

class_names = train_dataset.classes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
swin = models.swin_t(weights="IMAGENET1K_V1")

# replace classifier
swin.head = nn.Linear(swin.head.in_features, 4)

swin = swin.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(swin.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
EPOCHS = 30
best_acc = 0

for epoch in range(EPOCHS):

    swin.train()
    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = swin(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    # validation
    swin.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            outputs = swin(images)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(swin.state_dict(), "best_swin.pth")

In [ ]:
def evaluate(model, loader):
    model.eval()

    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [ ]:
y_true, y_pred, y_prob = evaluate(swin, test_loader)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')
bal_acc = balanced_accuracy_score(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob, multi_class='ovr')

cm = confusion_matrix(y_true, y_pred)

specificity = np.mean([
    cm[i,i] / (cm[:,i].sum()) for i in range(len(cm))
])

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Balanced Accuracy:", bal_acc)
print("Specificity:", specificity)
print("AUC:", auc)
print("Kappa:", kappa)

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)

plt.title("Swin Transformer")
plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
def tta_predict(model, images):

    outputs = []

    transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3])
    ]

    for t in transforms:
        aug = t(images)
        out = torch.softmax(model(aug), dim=1)
        outputs.append(out)

    return torch.mean(torch.stack(outputs), dim=0)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - MSW-EnsNet")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import cv2

def gradcam(model, image):

    model.eval()

    features = []
    gradients = []

    def forward_hook(module, input, output):
        features.append(output)

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    # EfficientNet last conv layer
    target_layer = model.features[-1]

    handle_f = target_layer.register_forward_hook(forward_hook)
    handle_b = target_layer.register_backward_hook(backward_hook)

    image = image.unsqueeze(0).to(device)

    output = model(image)
    pred = output.argmax(dim=1)

    loss = output[:, pred]
    model.zero_grad()
    loss.backward()

    grads = gradients[0]
    fmap = features[0]

    weights = grads.mean(dim=[2,3], keepdim=True)
    cam = (weights * fmap).sum(dim=1).squeeze().detach().cpu().numpy()

    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)

    cam = cv2.resize(cam, (224,224))

    handle_f.remove()
    handle_b.remove()

    return cam

In [ ]:
def rise(model, image, N=1000):

    model.eval()
    image = image.to(device)

    _, H, W = image.shape

    masks = torch.rand((N, 1, H, W)).to(device)

    preds = []

    with torch.no_grad():
        for m in masks:
            masked = image * m
            out = torch.softmax(model(masked.unsqueeze(0)), dim=1)
            preds.append(out)

    preds = torch.cat(preds)  # (N, num_classes)

    pred_class = preds.mean(dim=0).argmax()

    heatmap = (masks.squeeze() * preds[:, pred_class].view(-1,1,1)).mean(dim=0)

    heatmap = heatmap.cpu().numpy()
    heatmap = heatmap / (heatmap.max() + 1e-8)

    return heatmap

In [ ]:
def overlay_heatmap(image, heatmap, alpha=0.4):

    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    image = np.uint8(image * 255)

    overlay = cv2.addWeighted(image, 1-alpha, heatmap, alpha, 0)

    return overlay

In [ ]:
# Get sample image
img, label = test_dataset[0]

# Denormalize
img_np = img.permute(1,2,0).cpu().numpy()
mean = np.array([0.485,0.456,0.406])
std = np.array([0.229,0.224,0.225])

img_np = std * img_np + mean
img_np = np.clip(img_np, 0, 1)

# Grad-CAM
cam = gradcam(effnet, img)
grad_overlay = overlay_heatmap(img_np, cam)

# RISE
rise_map = rise(effnet, img)
rise_overlay = overlay_heatmap(img_np, rise_map)

# Plot
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img_np)
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(grad_overlay)
plt.title("Grad-CAM")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(rise_overlay)
plt.title("RISE")
plt.axis("off")

plt.show()